# Fine-tune pretrained DistilBERT for news retrieval

This experiment updates actual pretrained DistilBERT transformer weights. It does not train the separate projection-only model proposed in 3.3. The same sentence encoder is used for past-history text and candidate text, and gradients flow through both branches.

Start with title + subtitle, latest 10 clicks, an average-history profile, and 1,000 fixed validation requests. Compare a matched frozen checkpoint with one fine-tuned epoch before spending time on full validation. The default trains the final two transformer blocks and the encoder's existing dense module; earlier blocks remain frozen. Setting `trainable_transformer_layers=None` updates the whole encoder.

Important: the fine-tuned encoder requires newly generated candidate AND history vectors. This notebook generates them for each evaluated checkpoint. Existing frozen vectors are preserved. Gemini API embeddings cannot be fine-tuned by this local DistilBERT code.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
EXPERIMENT_ID = '3.4_finetuned_distilbert'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    print('Upload BOTH finetune_encoder.py and catalogue_pipeline.py from this experiment folder.')
    uploaded = files.upload()
    for name in ['finetune_encoder', 'catalogue_pipeline']:
        matches = [p for p in uploaded if Path(p).stem.startswith(name) and Path(p).suffix == '.py']
        if len(matches) != 1: raise ValueError(f'Upload exactly one {name}.py')
        (CODE_DIR / f'{name}.py').write_bytes(uploaded[matches[0]])
else:
    choices = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in choices if (p / 'finetune_encoder.py').is_file()), None)
    if CODE_DIR is None: raise FileNotFoundError('Open from this experiment folder or the repository root.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=1.26', 'pandas>=2.2',
    'pyarrow>=15', 'scikit-learn>=1.4', 'torch>=2.3', 'sentence-transformers>=3', 'huggingface-hub>=0.23'], check=True)
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE_DIR / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
cp = load_module('catalogue_pipeline')
ft = load_module('finetune_encoder')
print('GPU available:', torch.cuda.is_available())


## Configuration and source audit

Enable a Colab GPU. Start with the smaller shared catalogue now used by the group. The first pilot uses 10,000 training requests and one epoch; increase to 50,000 requests or more epochs only in a new run. Negative examples are drawn only from displayed-but-unclicked training articles, excluding every clicked article in the request. No unshown catalogue article is labeled as a negative.

The notebook resolves an exact base-model revision. To compare against a previously saved BERT encoder, copy its revision from `embedding_config.json`. Otherwise a fresh matched frozen baseline is generated from the revision resolved below. Text is serialized with real newline separators; the matched baseline is authoritative if old notebooks used different serialization/chunk settings.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
MODEL_NAME = 'sentence-transformers/distiluse-base-multilingual-cased-v2'
MODEL_REVISION = None  # Prefer the exact revision in your old BERT embedding_config.json.
if MODEL_REVISION is None:
    MODEL_REVISION = HfApi().model_info(MODEL_NAME).sha
RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + '_finetune_seed42'
WORK = CODE_DIR / 'outputs' / RUN_ID
CFG = {
    'experiment_id': EXPERIMENT_ID, 'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT), 'seed': 42,
    'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
    'train_requests': 10_000, 'validation_requests': 1000,
    'history_limit': 10, 'negatives_per_positive': 4,
    'trainable_transformer_layers': 2,  # None = all transformer layers.
    'epochs': 1, 'learning_rate': 1e-5, 'weight_decay': 0.01,
    'batch_size': 2, 'gradient_accumulation': 8,
    'chunk_batch_size': 16, 'encoding_article_batch': 64,
    'temperature': 0.05, 'amp': True,
    'similarity_device': 'auto', 'catalogue_batch_size': 4096,
    'max_index_gb': 20, 'hash_inputs': False, 'paths': {},
    # Optional ORIGINAL article files to supply missing historical text only.
    # This does not expand the candidate catalogue.
    'additional_text_catalogues': [],
}
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG['seed'])
print('Run:', WORK)
print('Exact encoder revision:', MODEL_REVISION)


In [ ]:
DATA = ft.prepare(CFG, WORK, cp)
display(pd.json_normalize(DATA['audit']))
print('Usable training examples:', len(DATA['examples']))
print('Candidate catalogue:', DATA['audit']['catalogue_articles'])
print('History text coverage:', DATA['audit']['history_coverage'])


## Load pretrained encoder and evaluate the matched frozen baseline

All candidate texts and available history texts use the same title/subtitle encoder and nonoverlapping token-chunk pooling. Every selected validation request searches its full eligible catalogue. Frozen and fine-tuned runs use identical request IDs, history length, mean-profile aggregation and publication eligibility.

Check history-text coverage before training. If the reduced catalogue omits old history articles, add original raw article files to `additional_text_catalogues` and create a fresh run. Existing vectors cannot supply raw text for gradient-based encoder updates.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODER = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION, device=DEVICE)
TRAINABLE = ft.configure_trainable(ENCODER, CFG['trainable_transformer_layers'])
print(TRAINABLE)
BASELINE = ft.evaluate_checkpoint(ENCODER, DATA, 'frozen', cp)
RESULTS = [BASELINE]
TRAINING_LOG = []
display(pd.json_normalize(BASELINE['metrics']))


## Fine-tune the pretrained encoder

The objective makes a user's mean-history representation score its clicked article above up to four observed unclicked alternatives. The same trainable encoder generates both sides. Actual candidate counts are used, so fewer-negative examples require no padding labels.

This is partial encoder fine-tuning by default, including the final two pretrained transformer blocks. No new user/article projection layers are created. Microbatches of two examples accumulate gradients to an effective eight-microbatch window. The chunk batch bounds each forward call, but activations within a training microbatch still require memory for backpropagation. If GPU memory is exhausted, use batch_size=1 or fewer trainable layers in a fresh run.

Each completed epoch saves encoder weights, regenerates its entire vector lookup, and evaluates catalogue Recall@200/nDCG@200. This is more expensive than frozen-vector experiments. Do not use test results for checkpoint selection. There is no automatic mid-epoch resume; preserve completed epoch checkpoints and the copied bundle.


In [ ]:
OPTIMIZER = torch.optim.AdamW(
    [p for p in ENCODER.parameters() if p.requires_grad],
    lr=CFG['learning_rate'], weight_decay=CFG['weight_decay'],
)
for epoch in range(1, CFG['epochs'] + 1):
    name = f'epoch_{epoch}'
    if (WORK / 'checkpoints' / name).exists():
        raise FileExistsError('Epoch checkpoint exists. Inspect saved results or start a fresh run; do not train twice into one epoch folder.')
    log = ft.train_epoch(ENCODER, OPTIMIZER, DATA, epoch)
    TRAINING_LOG.append(log)
    print(log)
    result = ft.evaluate_checkpoint(ENCODER, DATA, name, cp)
    torch.save({'epoch': epoch, 'optimizer': OPTIMIZER.state_dict(), 'config': CFG},
               WORK / 'checkpoints' / name / 'optimizer.pt')
    RESULTS.append(result)
    SUMMARY = ft.save_selected(DATA, RESULTS, TRAINING_LOG, cp)
    display(pd.json_normalize(result['metrics']))
SUMMARY = ft.save_selected(DATA, RESULTS, TRAINING_LOG, cp)


## Compare frozen and fine-tuned results

Selection uses validation Recall@200, then nDCG@200, preserving the earlier result in an exact tie. The frozen encoder can remain selected if fine-tuning does not improve it. Report that outcome honestly; training loss alone cannot establish retrieval improvement.


In [ ]:
comparison = pd.DataFrame([
    {'checkpoint': r['name'], 'recall@200': r['metrics']['known_click_recall@200'],
     'ndcg@200': r['metrics']['observed_click_ndcg@200'],
     'validation_requests': r['metrics']['evaluated_requests']}
    for r in RESULTS
])
display(comparison)
SELECTED = SUMMARY['selected']
print('Selected:', SELECTED['name'])
print('Selected encoder:', SELECTED['encoder_path'])
print('Selected embeddings:', SELECTED['embeddings_path'])


## Final held-out test comparison

Run only after fixing the training recipe and choosing checkpoints using validation. This compares the frozen encoder with the best fine-tuned checkpoint selected on validation. If the overall validation winner was frozen, the best fine-tuned checkpoint is still shown for the research comparison; test results do not change SUMMARY.

Both checkpoints use the same test requests and eligible catalogue, each with its own regenerated vectors and test-history profiles. Genuine labels are loaded into a separate temporary evaluation index; training/inference indexes remain unchanged. TEST_REQUESTS=None means all test requests and can take substantially longer than a sample. Ensure test requests are independent of validation, even if their time periods overlap.

This saves test_comparison.json before the Drive-copy cell, so it is included in the copied bundle. It evaluates metrics without exporting test shortlists; generate those later through the shared export path if required.


In [ ]:
import sqlite3
import tempfile
from contextlib import closing
import pyarrow.parquet as pq

TEST_REQUESTS = None  # All test requests. Use 1000 for a preliminary timing check.

# Select using VALIDATION results, before reading any test labels.
frozen = next(r for r in RESULTS if r['frozen_baseline'])
finetuned_runs = [r for r in RESULTS if not r['frozen_baseline']]
if not finetuned_runs:
    raise ValueError('Finish at least one fine-tuned epoch first.')
best_finetuned = ft.select_checkpoint(finetuned_runs)
checkpoints = [frozen, best_finetuned]

test_source = ft.resolve(CFG, 'features_test', 'feature_engineering/test_features.parquet')
test_history = ft.resolve(CFG, 'history_test', 'test/history.parquet')
source_info = cp.fingerprint(test_source, CFG.get('hash_inputs', False))
test_results = []

with tempfile.TemporaryDirectory(dir=WORK, prefix='heldout_test_') as directory:
    test_db = Path(directory) / 'evaluation.sqlite'
    index_info = cp.index_shared(test_source, 'test', test_db, CFG['seed'])

    # This separate, temporary EVALUATION index receives labels.
    # The inference/training indexes remain unchanged.
    with closing(sqlite3.connect(test_db)) as db:
        columns = ['user_id', 'impression_id', 'candidate_article_id', 'clicked']
        for batch in pq.ParquetFile(test_source).iter_batches(columns=columns, batch_size=8192):
            updates = []
            for row in batch.to_pylist():
                label = row['clicked']
                if label not in (0, 1):
                    raise ValueError('Test evaluation requires genuine, non-null binary click labels.')
                updates.append((int(label), cp.identity(row['user_id'], row['impression_id']),
                                int(row['candidate_article_id'])))
            db.executemany('UPDATE candidates SET clicked=? WHERE behaviour_id=? AND article_id=?', updates)
        db.commit()

    test_requests = list(cp.request_iter(test_db, TEST_REQUESTS))
    users = {int(r[1]) for r in test_requests}
    pairs = cp.budget(test_requests, DATA['catalogue'])['eligible_pairs']
    print(f'Test requests: {len(test_requests):,}; catalogue comparisons per checkpoint: {pairs:,}')

    for result in checkpoints:
        print('Evaluating:', result['name'], flush=True)
        # Each checkpoint must use ITS OWN regenerated vectors.
        vectors, embedding_info = cp.load_embeddings([Path(result['embeddings_path'])])
        profiles, coverage = cp.history_profiles(
            test_history, users, index_info['split_start'], vectors,
            history_limit=CFG['history_limit'], aggregation='mean_profile',
        )
        lookup = cp.SimilarityVectors(vectors, CFG.get('similarity_device', 'auto'), 256)
        specification = cp.model_spec({
            'history_limit': CFG['history_limit'], 'history_aggregation': 'mean_profile',
        })
        metrics = cp.evaluate_catalogue(
            specification, test_requests, test_db, DATA['catalogue'], profiles, lookup,
            CFG.get('catalogue_batch_size', 4096),
        )
        test_results.append({
            'checkpoint': result['name'], 'frozen_baseline': result['frozen_baseline'],
            'metrics': metrics, 'history_coverage': coverage,
            'embeddings_path': result['embeddings_path'],
        })
        del lookup, profiles, vectors

comparison_test = pd.DataFrame([
    {'checkpoint': r['checkpoint'],
     'test_recall@200': r['metrics']['known_click_recall@200'],
     'test_ndcg@200': r['metrics']['observed_click_ndcg@200'],
     'evaluated_test_requests': r['metrics']['evaluated_requests'],
     'requests_without_known_clicks': r['metrics']['requests_without_known_clicks']}
    for r in test_results
])
display(comparison_test)
cp.save_json(WORK / 'test_comparison.json', {
    'test_source': source_info,
    'test_request_ids': [r[0] for r in test_requests],
    'eligibility_policy': cp.POLICY,
    'history_limit': CFG['history_limit'], 'history_aggregation': 'mean_profile',
    'checkpoint_selection': 'Best fine-tuned checkpoint chosen using validation only; test does not change SUMMARY.',
    'results': test_results,
})
print('Saved:', WORK / 'test_comparison.json')


## Preserve the run on Drive

Copy checkpoints, regenerated embeddings, source audit and comparison results. Generated SQLite indexes are omitted. This can be large, so check space. Copy after completing the experiment; a local-only run can disappear when the Colab runtime ends. The original frozen embedding cache is never overwritten.


In [ ]:
import shutil
DESTINATION = DRIVE_ROOT / 'Experiments' / EXPERIMENT_ID / RUN_ID
if DESTINATION.exists():
    raise FileExistsError('Destination exists. Use a fresh copy suffix to preserve previous results.')
shutil.copytree(WORK, DESTINATION, ignore=shutil.ignore_patterns('indexes'))
print('Saved fine-tuning run:', DESTINATION)
print('Copied selected encoder:', DESTINATION / Path(SELECTED['encoder_path']).relative_to(WORK))
print('Copied selected embeddings:', DESTINATION / Path(SELECTED['embeddings_path']).relative_to(WORK))


## Use the selected checkpoint in the existing retrieval/export notebook

Point BOTH embedding inputs to the selected checkpoint's `text_embeddings.parquet`: it contains the union lookup needed for candidates and histories. The candidate catalogue continues to come from the shared article files and remains unchanged. The same lookup can be passed as both sources; the existing loader keeps the first copy of each ID.

Set history_limit=10 and history_aggregation='mean_profile', retain the same shared eligibility policy, use a fresh run ID, and evaluate/export the group's agreed request scope. Record the selected encoder path, original model revision and fine-tuned status in embedding_source. For validation decisions, do not select a mean-profile-trained encoder by silently replacing its aggregation with top-k. Any such comparison is a separate experiment.

The selected vectors can also represent test-history articles already present in the declared text lookups. Only after fixing all settings should you export test recommendations and use the separate held-out test evaluation cell. Test labels are not loaded by this fine-tuning notebook.
